In [18]:
from datasets import load_dataset
import json

In [19]:
# load dataset from hugging-face
dataset = load_dataset("hate-speech-portuguese/hate_speech_portuguese")

In [20]:
# view dataset features info
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'],
        num_rows: 5670
    })
})


In [21]:
# Removes unused columns
dataset = dataset.remove_columns(['hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'])

In [22]:
# view dataset feature info before remove unused columns
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5670
    })
})


In [23]:
# split dataset in train / test
train_dataset = dataset["train"]

split = train_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split["train"]
test_dataset = split["test"]

In [24]:
print(train_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 4536
})


In [25]:
# view sample data
print("text:\n")
print(train_dataset["text"][0])
print("label:\n")
print(train_dataset["label"][0])

text:

Desde 8 de Nov, dia da eleição, o mercado de ações cresceu $3.2 trilhões e a confiança do consumidor está em uma al _ https://t.co/7nUXa3vpuf
label:

0


In [26]:
# remove special "\n" character
def removeN(example):
    example["text"].replace("\n", " ")
    return example

In [27]:
# map function ton remove "\n" character
train_dataset = train_dataset.map(removeN)
test_dataset = test_dataset.map(removeN)

In [28]:
def labelChange(example):
    example["label_text"] = "No Hate Speech" if example["label"]==0 else "Hate Speech"
    return example

In [29]:
# map label names
train_dataset = train_dataset.map(labelChange)
test_dataset = test_dataset.map(labelChange)

In [30]:
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 4536
})
Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 1134
})


In [31]:
def dataset_to_jsonl(dataset, file_name):
    with open(file_name, "w", encoding="utf-8") as f:
        for example in dataset:
            json_obj = {
                "input": example["text"],
                "output": example["label_text"]
            }
            f.write(json.dumps(json_obj, ensure_ascii=False)+ "\n")

In [32]:
dataset_to_jsonl(train_dataset, "train.jsonl")
dataset_to_jsonl(test_dataset, "test.jsonl")